---
technologies: "FastAPI"
category: "Choice and use of technology"
difficulty: "Intermediate"
---

# FastAPI 

## Used material

1. <span id="used-material-1"></span> [Run Redis on Docker](https://redis.io/docs/latest/operate/oss_and_stack/install/install-stack/docker/)

2. <span id="used-material-2"></span> [FastAPI pip package](https://pypi.org/project/fastapi/)

3. <span id="used-material-3"></span> [Uvicorn pip package](https://pypi.org/project/uvicorn/)

4. <span id="used-material-4"></span> [Redis pip package](https://pypi.org/project/redis/)

5. <span id="used-material-5"></span> [FastAPI tutorial](https://fastapi.tiangolo.com/tutorial/)

6. <span id="used-material-6"></span> [FastAPI server](https://fastapi.tiangolo.com/deployment/manually/#install-the-server-program)

7. <span id="used-material-7"></span> [Running Uvicorn](https://www.uvicorn.org/#uvicornrun)

## Why use FastAPI?

FastAPI was chosen as the default frontend framework for the following reasons:

- One of the most popular web frameworks for Python (mature)

- Provides an easy way to create and modify REST API routes (abstracted)

- Usable in any place where Python can be used (interoperable)

These enable us to use FastAPI to create interfaces that machines and humans can easily use, enabling flexible integration automation.
 
## How to use FastAPI?

Assuming we have a running Redis container [(1)](#used-material-1), we can setup the FastAPI front end by setting up a venv |[(2)](#used-material-2), [(3)](#used-material-3), [(4)](#used-material-4)| and executing the [run](./fastapi/run_fastapi.py) file in the following way |[(5)](#used-material-5), [(6)](#used-material-6), [(7)](#used-material-7)|:


```
cd fastapi
python3 -m venv fastapi_venv
source fastapi_venv/bin/activate
pip install -r packages.txt
python3 run_fastapi.py
```

We can now check the available routes at http://localhost:7500/docs. To test the [logs](./fastapi/routes/general.py) route and see the [log](./fastapi/logs/frontend-logs.txt) file do, the following:

1. Click the get /general/logs window

2. Click the try it out and execute buttons

3. Scroll down to see the response body

Now, we have abstracted FastAPI route interactions with [integration](./integration.py), which we will use to store parameters into Redis in the following way:

In [1]:
%run ./integration.py

In [2]:
swift_parameters = {
    'pre-auth-url': 'test',
    'pre-auth-token': 'test',
    'user-domain-name': 'test',
    'project-domain-name': 'test',
    'project-name': 'test',
    'auth-version': 'test'
}
bucket_parameters = {
    'target': 'submitter',
    'prefix': 'mlch',
    'user': 'user@example.com'
}
storage_parameters = {
    'cache-location': 'local',
    'cache': {
        'local': {
            'system': 'redis',
            'user': 'part-2',
            'target': 'frontend',
            'redis-endpoint': '127.0.0.1',
            'redis-port': '6379',
            'redis-db': '0',
        }
    }
}

In [3]:
configuration_dictionary = {
    'swift-parameters': swift_parameters,
    'bucket-parameters': bucket_parameters,
    'storage-parameters': storage_parameters
}

FastAPI routes use Pydantic models to validate inputs, which in this case for the [setup](./fastapi/routes/setup.py) route is [parameters](./fastapi/functions/models/parameters.py) model. 

In [4]:
route_code, route_text = integration_request_route(
    address = '127.0.0.1',
    port = '7500',
    route_type = '',
    route_name = 'setup',
    path_replacers = {},
    path_names = [],
    route_input = configuration_dictionary,
    timeout = 10
)
print(route_code)

Used route: POST:/setup/config
200


In [5]:
route_text

{'output': True}

We can use [caching](./caching.py) functions to get the stored parameters: 

In [6]:
%run ./caching.py

In [7]:
cache_location = storage_parameters['cache-location']
cache_parameters = storage_parameters['cache'][cache_location]

cache_key = caching_generate_key(
    static = True,
    user = cache_parameters['user'],
    target = cache_parameters['target'],
    group = 'dict'
)

In [8]:
print(cache_key)

cache:part-2:frontend:dict


In [9]:
fetched_nested_dict = caching_get_dict(
    cache_parameters = cache_parameters,
    cache_key = cache_key
)

Testing connection: 127.0.0.1:6379:0
Using key: cache:part-2:frontend:dict


In [10]:
fetched_nested_dict

{'swift-parameters': {'pre-auth-url': 'test',
  'pre-auth-token': 'test',
  'user-domain-name': 'test',
  'project-domain-name': 'test',
  'project-name': 'test',
  'auth-version': 'test'},
 'bucket-parameters': {'target': 'submitter',
  'prefix': 'mlch',
  'user': 'user@example.com'},
 'storage-parameters': {'cache-location': 'local',
  'cache': {'local': {'system': 'redis',
    'user': 'part-2',
    'target': 'frontend',
    'redis-endpoint': '127.0.0.1',
    'redis-port': '6379',
    'redis-db': '0'}}}}

You can see what actions the server did by checking the [log](./fastapi/logs/frontend-logs.txt) in the following way:

In [11]:
route_code, route_text = integration_request_route(
    address = '127.0.0.1',
    port = '7500',
    route_type = '',
    route_name = 'logs',
    path_replacers = {},
    path_names = [],
    route_input = {},
    timeout = 10
)
print(route_code)

Used route: GET:/general/logs
200


In [12]:
route_text

{'output': {'logs': ['2026-03-24 12:36:40,262 - INFO - submitter-frontend - Creating FastAPI instance',
   '2026-03-24 12:36:40,262 - INFO - submitter-frontend - FastAPI created',
   '2026-03-24 12:36:40,262 - INFO - submitter-frontend - Connecting to Redis',
   '2026-03-24 12:36:40,286 - INFO - submitter-frontend - Redis connected',
   '2026-03-24 12:36:40,286 - INFO - submitter-frontend - Importing routes',
   '2026-03-24 12:36:40,298 - INFO - submitter-frontend - Including routers',
   '2026-03-24 12:36:40,299 - INFO - submitter-frontend - Routes implemented',
   '2026-03-24 12:36:40,299 - INFO - submitter-frontend - Frontend ready',
   '2026-03-24 12:36:58,172 - INFO - submitter-frontend - Post configuration',
   '2026-03-24 12:36:58,172 - INFO - submitter-frontend - cache:part-2:frontend:dict',
   '2026-03-24 12:36:58,173 - INFO - submitter-frontend - Testing connection: 127.0.0.1:6379:0',
   '2026-03-24 12:36:58,177 - INFO - submitter-frontend - Using key: cache:part-2:frontend:d

## FastAPI file structure

When you check the files inside the fastapi folder, you will notice the following structure:

- [packages](./fastapi/packages.txt)
- [run_fastapi](./fastapi/run_fastapi.py)
- [setup_fastapi](./fastapi/setup_fastapi.py)
- logs:
    - [frontend-logs](./fastapi/logs/frontend-logs.txt)
- routes:
    - [general](./fastapi/routes/general.py)
    - [setup](./fastapi/routes/setup.py)
- functions:
    - models:
        - [parameters](./fastapi/functions/models/parameters.py)
    - platforms:
        - logger:
            - [setup](./fastapi/functions/platforms/logger/setup.py)
            - [use](./fastapi/functions/platforms/logger/use.py)
        - redis:
            - [setup](./fastapi/functions/platforms/redis/setup.py)
            - [use](./fastapi/functions/platforms/redis/use.py)

The purpose of this structure in this and following cases is to enable easier understandability and development by dividing the necessery code into their smallest practical form using abstraction layers. This enables us to clearly divide the responsiblity in return for more complex file structures. 

## Important parts of FastAPI

The most important parts to keep eye on when trying to understand or develop a FastAPI application are:

- State = Storing objects into a instance for application level context (see use in [setup](./fastapi/functions/platforms/logger/setup.py) row 16)
- APIRouter = Smaller instances that can consist of multiple routes (see use in [setup](./fastapi/functions/platforms/logger/setup.py) row 40)
- Prefix = Group path for the APIRouter (see use in [general](./fastapi/routes/general.py) row 5)
- Route = Code server executes for requests with a specific URL and HTTP method (see use in [general](./fastapi/routes/general.py) rows 7-18)
- Route parameters = Request object, path parameters and provided parameters that code can use (see use in [setup](./fastapi/routes/setup.py) rows 8-36)

---